# Data Augmentation with Generative Adversarial Networks (GANs)
In this notebook, we will implement and train a GAN for data augmentation. The objective is to train a generator that can generate realistic samples of benign network traffic.

We will use a dataset of benign traffic from the CIC-DDoS2019 dataset (https://www.unb.ca/cic/datasets/ddos-2019.html).
The network traffic has been previously pre-processed in a way that packets are grouped in bi-directional traffic flows using the 5-tuple (source IP, destination IP, source Port, destination Port, protocol). Each flow is represented with 21 packet-header features computed from max 1000 packets:

| Feature nr.         | Feature Name |
|---------------------|---------------------|
| 00 | timestamp (mean IAT) | 
| 01 | packet_length (mean)| 
| 02 | IP_ttl (mean) |
| 03 | IP_flags_df (sum) |
| 04 | IP_flags_mf (sum) |
| 05 | IP_flags_rb (sum) | 
| 06 | IP_frag_off (sum) |
| 07 | TCP_length (mean) |
| 08 | TCP_flags_ack (sum) |
| 09 | TCP_flags_cwr (sum) |
| 10 | TCP_flags_ece (sum) |
| 11 | TCP_flags_fin (sum) |
| 12 | TCP_flags_push (sum) |
| 13 | TCP_flags_reset (sum) |
| 14 | TCP_flags_syn (sum) |
| 15 | TCP_flags_urg (sum) |
| 16 | TCP_window_size (mean) |
| 17 | UDP_length (mean) |
| 18 | ICMP_type (mean) |
| 19 | ICMP_code (mean) |
| 20 | Packets (counter)|

In [ ]:
# Author: Roberto Doriguzzi-Corin
# Project: Course on Network Intrusion and Anomaly Detection with Machine Learning
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#   http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LeakyReLU, ReLU,BatchNormalization, Reshape, Flatten, Input, Conv2D,GlobalMaxPooling2D
from scikeras.wrappers import KerasRegressor
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV
from tensorflow.keras.optimizers import Adam, SGD
from scipy.stats import uniform, randint
import numpy as np
import matplotlib.pyplot as plt
import h5py
from util_functions import *

SEED = 0

DATASET_FOLDER = "./DOS2019_Benign_Flatten/"

X_train, _ = load_dataset(DATASET_FOLDER + "/*" + '10t-1000n-DOS2019-benign-train.hdf5')
X_val, _ = load_dataset(DATASET_FOLDER + "/*" + '10t-1000n-DOS2019-benign-val.hdf5')
X_test, _ = load_dataset(DATASET_FOLDER + "/*" + '10t-1000n-DOS2019-benign-test.hdf5')

In [ ]:
# Function to save generated images in HDF5 format
def save_generated_images(generator, latent_dim=100, save_path='./DOS2019_Benign_Flatten/10t-1000n-DOS2019-fake-benign-train.hdf5 ', num_samples=1000):
    noise = np.random.normal(0, 1, (num_samples, latent_dim))
    generated_images = generator.predict(noise)
    labels = np.zeros((generated_images.shape[0],))

    hf = h5py.File(save_path, 'w')
    hf.create_dataset('set_x', data=np.squeeze(generated_images))
    hf.create_dataset('set_y', data=labels)
    hf.close()

# Evaluation of the discriminator's accuracy on real and fake samples

In [ ]:
def evaluate_discriminator_accuracy(generator, discriminator, real_data, latent_dim=100):

    num_samples = real_data.shape[0]
    # Generate fake images using the generator
    noise = np.random.normal(0, 1, (num_samples, latent_dim))
    generated_images = np.squeeze(generator.predict(noise))

    # Label real and fake images
    real_labels = np.ones((num_samples, 1))
    fake_labels = np.zeros((num_samples, 1))

    # Evaluate the discriminator on real and generated data
    real_accuracy = discriminator.evaluate(real_data, real_labels, verbose=0)[1]
    fake_accuracy = discriminator.evaluate(generated_images, fake_labels, verbose=0)[1]

    # Calculate overall accuracy
    overall_accuracy = 0.5 * (real_accuracy + fake_accuracy)

    print(f"Real Accuracy: {real_accuracy * 100:.2f}%")
    print(f"Fake Accuracy: {fake_accuracy * 100:.2f}%")
    print(f"Overall Discriminator Accuracy: {overall_accuracy * 100:.2f}%")

    return overall_accuracy, real_accuracy, fake_accuracy

# GAN Architecture

In [ ]:
# Generator function
def build_generator(latent_dim, output_shape):
    model = Sequential()
    model.add(Input(shape=(latent_dim,)))
    model.add(Dense(128))
    model.add(ReLU())
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(256))
    model.add(ReLU())
    model.add(BatchNormalization(momentum=0.8))
    model.add(Dense(output_shape[0], activation='sigmoid'))
    model.add(Reshape(output_shape))
    return model

# Discriminator function
def build_discriminator(input_shape):
    model = Sequential()
    model.add(Input(shape=input_shape))
    model.add(Flatten())
    # model.add(Dense(256))
    # model.add(ReLU())
    model.add(Dense(128))
    model.add(ReLU())
    model.add(Dense(1, activation='sigmoid'))
    return model

# GAN function
def build_gan(generator, discriminator):
    discriminator.trainable = False
    model = Sequential()
    model.add(generator)
    model.add(discriminator)
    return model

# Build the GAN
We will build a GAN with a generator and a discriminator.

In [ ]:
# Hyperparameters
latent_dim = 100
img_shape = (X_train.shape[1], 1)

# Build and compile the discriminator
discriminator = build_discriminator(img_shape)
discriminator.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5), metrics=['accuracy'])

# Build and compile the generator
generator = build_generator(latent_dim, img_shape)
generator.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))

# Build and compile the GAN
discriminator.trainable = False
gan = build_gan(generator, discriminator)
gan.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))

# Training the GAN
The generator is trained until the accuracy of the discriminator on the fake samples is below a certain threshold. This indicates high quality of the generator's output. 

In [ ]:
# Training loop
epochs = 1000
patience = 0
batch_size = 512
half_batch = batch_size // 2
lowest_d_fake_acc = float('inf')
best_generator = generator
g_loss_history = []
d_loss_history = []
d_overall_accuracy_history = []
d_fake_accuracy_history = []
d_real_accuracy_history = []
TARGET_FAKE_ACCURACY = 0.05


for epoch in range(epochs):
    # ---------------------
    # Train the discriminator
    # ---------------------

    idx = np.random.randint(0, X_train.shape[0], half_batch)
    imgs = X_train[idx]

    noise = np.random.normal(0, 1, (half_batch, latent_dim))
    gen_imgs = generator.predict(noise) # generate new fake samples from random noise

    discriminator.trainable = True # Unfreeze the discriminator's weights during discriminator training
    d_loss_real = discriminator.train_on_batch(imgs, np.ones((half_batch, 1))) # label 1 for real samples
    d_loss_fake = discriminator.train_on_batch(gen_imgs, np.zeros((half_batch, 1))) # label 0 for fake samples
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    # ---------------------
    # Train the generator
    # ---------------------
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    valid_y = np.array([1] * batch_size) # label 1 for fake samples

    discriminator.trainable = False # Freeze the discriminator's weights during generator training
    g_loss = gan.train_on_batch(noise, valid_y) 

    # Evaluate progress after some warm-up epochs
    print(f"Epoch {epoch}, D Loss: {d_loss[0]}, G Loss: {g_loss}")
    overall_accuracy, real_accuracy, fake_accuracy = evaluate_discriminator_accuracy(generator, discriminator, real_data=X_train, latent_dim=latent_dim)
    g_loss_history.append(g_loss)
    d_loss_history.append(d_loss[0])
    d_overall_accuracy_history.append(overall_accuracy)
    d_real_accuracy_history.append(real_accuracy)
    d_fake_accuracy_history.append(fake_accuracy)

    if fake_accuracy <= TARGET_FAKE_ACCURACY and epoch > 100:  # Ensure we have enough epochs before stopping
        break

save_generated_images(generator)

#plot the loss and accuracy history
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(g_loss_history, color='red', label='Generator Loss')
plt.plot(d_loss_history, color='blue', label='Discriminator Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(d_overall_accuracy_history, color='red', label='Overall Accuracy')
plt.plot(d_real_accuracy_history, color='blue', label='Real Accuracy')
plt.plot(d_fake_accuracy_history, color='yellow', label='Fake Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Discriminator Accuracy')
plt.legend()


# Test the quality of the fake samples
We train an autoencoder with the real benign samples and then we test the reconstruction error of the autoencoder on the fake samples. If the reconstruction error is low, it means that the fake samples are similar to the real samples and therefore they are of high quality.

In [ ]:
def create_model(hidden_units=10,coding_layer_size=2, learning_rate = 0.001, optimizer=SGD):
    stacked_encoder = Sequential(name='Encoder',layers=[Input(shape=(X_train.shape[1],)), 
                              Dense(hidden_units, activation="relu"), 
                              Dense(coding_layer_size, activation="relu")
                              ]) 

    stacked_decoder = Sequential(name='Decoder',layers=[ 
                            Input(shape=(coding_layer_size,)),
                            Dense(hidden_units, activation="relu"), 
                            Dense(X_train.shape[1], activation="sigmoid") 
                            ]) 
    stacked_ae = Sequential([stacked_encoder, stacked_decoder])

    # Compile the model
    stacked_ae.compile(optimizer=optimizer(learning_rate=learning_rate), loss='mean_squared_error')
    print (stacked_encoder.summary())
    print (stacked_decoder.summary())
    return stacked_ae

In [ ]:
# Train an autoencoder on the real benign samples
autoencoder = create_model(hidden_units=64, coding_layer_size=32, learning_rate=0.001, optimizer=Adam)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
autoencoder.fit(X_train, X_train, epochs=100, batch_size=512, validation_data=(X_val, X_val), callbacks=[early_stopping])   

In [ ]:
# Test the model on real test samples
reconstructed_test = autoencoder.predict(X_test)
mse_test = np.mean(np.square(X_test - reconstructed_test), axis=1)
print(f"Mean Squared Error on Test Set: {np.mean(mse_test)}")

# Use the generator to create fake samples
noise = np.random.normal(0, 1, (1000, latent_dim))
fake_samples = np.squeeze(generator.predict(noise))

# Test the autoencoder on the fake samples
reconstructed_fake = autoencoder.predict(fake_samples)
mse_fake = np.mean(np.square(fake_samples - reconstructed_fake), axis=1)
print(f"Mean Squared Error on Fake Samples: {np.mean(mse_fake)}")